# E-commerce Recommendation Modeling

Implicit-feedback candidate generation, co-visitation features, CatBoost ranking, MLflow logging, and serving artifacts for e-commerce recommendations.

Outputs were stripped for public release. Re-run the notebook after configuring the required private data access or local artifact paths.


In [ ]:
import os
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from IPython.display import display

import mlflow
from sklearn.preprocessing import LabelEncoder
from implicit.als import AlternatingLeastSquares
from catboost import CatBoostClassifier, Pool

pd.set_option("display.max_columns", None)


In [ ]:
DATA_DIR = "../data"

events = pd.read_csv(f"{DATA_DIR}/events.csv")

# Type conversion
if "timestamp" in events.columns:
    events["timestamp"] = pd.to_numeric(events["timestamp"], errors="coerce")
    events["ts"] = pd.to_datetime(events["timestamp"], unit="ms", errors="coerce")
    events["date"] = events["ts"].dt.date

if "transactionid" in events.columns:
    events["transactionid"] = pd.to_numeric(events["transactionid"], errors="coerce").astype("Int64")

print("shape events", events.shape)
print(events.head(3).to_string(index=False))
print(events.dtypes.to_string())

n_users = events["visitorid"].nunique()
n_items = events["itemid"].nunique()
print("n_users", n_users, "n_items", n_items)

print("events by type")
print(events["event"].value_counts(dropna=False))


In [ ]:
# Events by day
by_day = (
    events.groupby("date")
          .agg(events=("event","count"), users=("visitorid","nunique"))
          .reset_index()
)
display(by_day.head())

# See the project README for context.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
events["event"].value_counts().plot(kind="bar", ax=ax[0], title="Event distribution")
ax[1].plot(by_day["date"], by_day["events"])
ax[1].set_title("Events by day")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:
paths = [f"{DATA_DIR}/item_properties_part1.csv",
         f"{DATA_DIR}/item_properties_part2.csv"]

usecols = ["timestamp","itemid","property","value"]
chunksize = 1_000_000
parts = []

for p in paths:
    for ch in pd.read_csv(p, usecols=usecols, chunksize=chunksize):
        cat = ch[ch["property"] == "categoryid"].copy()
        if len(cat) == 0:
            continue
        cat["timestamp"] = pd.to_numeric(cat["timestamp"], errors="coerce")
        parts.append(cat[["timestamp","itemid","value"]])

item_cat = pd.concat(parts, ignore_index=True)
item_cat = item_cat.dropna(subset=["timestamp"])
item_cat["ts"] = pd.to_datetime(item_cat["timestamp"], unit="ms", errors="coerce")

# See the project README for context.
last_cat = (
    item_cat.sort_values(["itemid","ts"])
            .groupby("itemid")
            .tail(1)[["itemid","value"]]
            .rename(columns={"value":"categoryid"})
)
last_cat["categoryid"] = pd.to_numeric(last_cat["categoryid"], errors="coerce").astype("Int64")


In [ ]:
print("items with category", len(last_cat))
last_cat.head()


In [ ]:
os.makedirs("../data/processed", exist_ok=True)
last_cat.to_parquet("../data/processed/items_categories.parquet", index=False)


In [ ]:
cat_tree = pd.read_csv(f"{DATA_DIR}/category_tree.csv")
ct = cat_tree.copy()
ct.columns = [c.lower() for c in ct.columns]

ct = ct.rename(columns={"parentid":"parent_category","categoryid":"child_category"})


In [ ]:
# See the project README for context.
events_cat = events.merge(last_cat, on="itemid", how="left")

evt_by_cat = (
    events_cat.groupby(["categoryid","event"]).size().unstack(fill_value=0)
              .assign(total=lambda df: df.sum(1))
              .sort_values("total", ascending=False)
              .head(10)
)
evt_by_cat


In [ ]:
# See the project README for context.
evt = events["event"].value_counts()
views = int(evt.get("view", 0))
adds  = int(evt.get("addtocart", 0))
txns  = int(evt.get("transaction", 0))

print("views", views, "adds", adds, "txns", txns)
print("addtocart rate", round(adds / views, 4))
print("purchase from addtocart", round(txns / max(adds,1), 4))


In [ ]:
# See the project README for context.
events_cat = events.merge(last_cat, on="itemid", how="left")
by_cat = (
    events_cat.groupby(["categoryid","event"]).size()
              .unstack(fill_value=0)
              .rename_axis(None, axis=1)
              .reset_index()
)
if "view" not in by_cat:      by_cat["view"] = 0
if "addtocart" not in by_cat: by_cat["addtocart"] = 0
if "transaction" not in by_cat: by_cat["transaction"] = 0

by_cat["atc_rate"] = by_cat["addtocart"] / by_cat["view"].replace(0, np.nan)
by_cat["buy_from_atc"] = by_cat["transaction"] / by_cat["addtocart"].replace(0, np.nan)

min_views = 5000
top_atc = by_cat.query("view >= @min_views").sort_values("atc_rate", ascending=False).head(10)
top_buy = by_cat.query("addtocart >= 500").sort_values("buy_from_atc", ascending=False).head(10)

print("Step completed")
top_atc[["categoryid","view","addtocart","atc_rate"]]


In [ ]:
print("Step completed")
top_buy[["categoryid","addtocart","transaction","buy_from_atc"]]


In [ ]:
# See the project README for context.
daily = (
    events_cat.groupby(["date","event"]).size()
              .unstack(fill_value=0)
              .reset_index()
              .sort_values("date")
)
for col in ["view","addtocart","transaction"]:
    if col not in daily: daily[col] = 0

daily["atc_rate"] = daily["addtocart"] / daily["view"].replace(0, np.nan)

fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].plot(daily["date"], daily["addtocart"])
ax[0].set_title("addtocart  ")
ax[1].plot(daily["date"], daily["atc_rate"])
ax[1].set_title("atc_rate  ")
fig.autofmt_xdate()
plt.tight_layout(); plt.show()


In [ ]:
# See the project README for context.
u_cnt = events.groupby("visitorid").size().rename("events_per_user")
i_cnt = events.groupby("itemid").size().rename("events_per_item")

print("users", len(u_cnt), "items", len(i_cnt))
print("median events per user", int(u_cnt.median()), "p95", int(u_cnt.quantile(0.95)))
print("median events per item", int(i_cnt.median()), "p95", int(i_cnt.quantile(0.95)))


In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
ax[0].hist(u_cnt, bins=50, log=True)
ax[0].set_title("   ")
ax[1].hist(i_cnt, bins=50, log=True)
ax[1].set_title("   ")
plt.tight_layout(); plt.show()

# See the project README for context.
top_items = (
    events.query("event == 'view'")
          .groupby("itemid").size().rename("views")
          .sort_values(ascending=False).head(10).reset_index()
)
top_items


In [ ]:
# Event weights
w = {"view": 1, "addtocart": 3, "transaction": 5}

events_prep = (
    events[["visitorid","itemid","event","ts"]]
    .dropna(subset=["ts"])
    .copy()
)
events_prep["strength"] = events_prep["event"].map(w).fillna(0).astype(np.int16)

# See the project README for context.
SPLIT_DATE = pd.Timestamp("2015-09-01")

train_mask = events_prep["ts"] < SPLIT_DATE
events_train = events_prep.loc[train_mask].copy()
events_test  = events_prep.loc[~train_mask].copy()

print("train rows", len(events_train), "test rows", len(events_test))

# See the project README for context.
ui_train = (
    events_train
    .groupby(["visitorid","itemid"], as_index=False)["strength"].sum()
)

print("ui_train pairs", len(ui_train))

# See the project README for context.
test_atc = (
    events_test.query("event == 'addtocart'")[["visitorid","itemid"]]
    .drop_duplicates()
)
print("test addtocart pairs", len(test_atc))


In [ ]:
# See the project README for context.
u_cnt = ui_train.groupby("visitorid")["itemid"].nunique()
i_cnt = ui_train.groupby("itemid")["visitorid"].nunique()
good_users = u_cnt[u_cnt >= 2].index
good_items = i_cnt[i_cnt >= 3].index

before = len(ui_train)
ui_train = ui_train[
    ui_train["visitorid"].isin(good_users) &
    ui_train["itemid"].isin(good_items)
].copy()
print("after filters", len(ui_train), "removed", before - len(ui_train))


In [ ]:
# See the project README for context.
allowed_users = ui_train["visitorid"].unique()
allowed_items = ui_train["itemid"].unique()

# See the project README for context.
events_test = events_test[
    events_test["visitorid"].isin(allowed_users) &
    events_test["itemid"].isin(allowed_items)
].copy()

test_atc = test_atc[
    test_atc["visitorid"].isin(allowed_users) &
    test_atc["itemid"].isin(allowed_items)
].copy()

# See the project README for context.
from sklearn.preprocessing import LabelEncoder
u_enc = LabelEncoder().fit(allowed_users)
i_enc = LabelEncoder().fit(allowed_items)

# See the project README for context.
ui_train["user_id_enc"] = u_enc.transform(ui_train["visitorid"])
ui_train["item_id_enc"] = i_enc.transform(ui_train["itemid"])

# See the project README for context.
events_test["user_id_enc"] = u_enc.transform(events_test["visitorid"])
events_test["item_id_enc"] = i_enc.transform(events_test["itemid"])
test_atc["user_id_enc"]    = u_enc.transform(test_atc["visitorid"])
test_atc["item_id_enc"]    = i_enc.transform(test_atc["itemid"])

# See the project README for context.
items = pd.DataFrame({"item_id": i_enc.classes_})
items = items.merge(last_cat.rename(columns={"itemid":"item_id"}), on="item_id", how="left")
items = items.rename(columns={"categoryid":"category_id"})
items["item_id_enc"] = i_enc.transform(items["item_id"])


In [ ]:
# See the project README for context.
ui_train.to_parquet("../data/processed/ui_train.parquet", index=False)
events_test.to_parquet("../data/processed/events_test.parquet", index=False)
test_atc.to_parquet("../data/processed/test_atc.parquet", index=False)
items.to_parquet("../data/processed/items.parquet", index=False)

print("users enc", ui_train["user_id_enc"].nunique(), "items enc", ui_train["item_id_enc"].nunique())
print("events_test shape", events_test.shape, "test_atc shape", test_atc.shape, "items shape", items.shape)


In [ ]:
# See the project README for context.
n_users = ui_train["user_id_enc"].max() + 1
n_items = ui_train["item_id_enc"].max() + 1

user_item = sp.csr_matrix(
    (ui_train["strength"].astype(np.float32),
     (ui_train["user_id_enc"], ui_train["item_id_enc"])),
    shape=(n_users, n_items)
)

# ALS
als = AlternatingLeastSquares(
    factors=64, iterations=25, regularization=0.05, random_state=0
)
als.fit(user_item)

print("matrix", user_item.shape, "nnz", user_item.nnz)


In [ ]:
# Generate top-N recommendations for test users
test_users = sorted(events_test["user_id_enc"].unique())
TOPN = 100

item_ids_enc, scores = als.recommend(
    test_users,
    user_item[test_users],
    N=TOPN,
    filter_already_liked_items=True
)

als_recs = pd.DataFrame({
    "user_id_enc": np.repeat(test_users, TOPN),
    "item_id_enc": item_ids_enc.flatten(),
    "score": scores.flatten()
})

print("recs shape", als_recs.shape)
als_recs.head()


In [ ]:
def eval_at_k(recs: pd.DataFrame, truth: pd.DataFrame, k: int = 5):
    # See the project README for context.
    r = recs.sort_values(["user_id_enc","score"], ascending=[True, False]) \
            .groupby("user_id_enc").head(k) \
            .groupby("user_id_enc")["item_id_enc"].apply(list)

    t = truth.groupby("user_id_enc")["item_id_enc"].apply(set)

    users = sorted(set(r.index) & set(t.index))
    precs, recalls, ndcgs, hits = [], [], [], []

    for u in users:
        rec_list = r[u]
        true_set = t[u]
        if not rec_list:
            continue
        hit = len(set(rec_list) & true_set)
        precs.append(hit / len(rec_list))
        if len(true_set) > 0:
            recalls.append(hit / len(true_set))
            # NDCG with binary relevance
            dcg = 0.0
            for i, it in enumerate(rec_list, start=1):
                if it in true_set:
                    dcg += 1 / np.log2(i + 1)
            ideal = min(k, len(true_set))
            idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal + 1))
            ndcgs.append(dcg / idcg if idcg > 0 else np.nan)
        hits.append(1 if hit > 0 else 0)

    return {
        "users_eval": len(users),
        f"precision@{k}": float(np.mean(precs)) if len(precs) else np.nan,
        f"recall@{k}": float(np.mean(recalls)) if len(recalls) else np.nan,
        f"ndcg@{k}": float(np.nanmean(ndcgs)) if len(ndcgs) else np.nan,
        "hit_rate": float(np.mean(hits)) if len(hits) else np.nan,
    }

m5  = eval_at_k(als_recs, test_atc, k=5)
m20 = eval_at_k(als_recs, test_atc, k=20)

# See the project README for context.
cov_items = (als_recs.groupby("user_id_enc").head(100)["item_id_enc"].nunique()
             / n_items)

print("users in eval", m5["users_eval"])
print("precision@5", round(m5["precision@5"], 4),
      "recall@5", round(m5["recall@5"], 4),
      "ndcg@5", round(m5["ndcg@5"], 4),
      "hit_rate", round(m5["hit_rate"], 4))
print("precision@20", round(m20["precision@20"], 4),
      "recall@20", round(m20["recall@20"], 4),
      "ndcg@20", round(m20["ndcg@20"], 4))
print("coverage@100", round(cov_items, 4))

# Save ALS candidates
als_recs.to_parquet("../data/processed/als_recommendations.parquet", index=False)


In [ ]:
# Build item similarities from co-occurring users
ui = ui_train[["visitorid","user_id_enc","item_id_enc"]].copy()

pairs = ui.merge(ui, on="visitorid")
pairs = pairs[pairs["item_id_enc_x"] != pairs["item_id_enc_y"]]

covis = (
    pairs.groupby(["item_id_enc_x","item_id_enc_y"])
         .size()
         .reset_index(name="score")
)

covis = covis.sort_values(["item_id_enc_x","score"], ascending=[True, False])
sim_items = covis.groupby("item_id_enc_x").head(50).reset_index(drop=True)
sim_items = sim_items.rename(columns={"item_id_enc_x":"item_id_enc",
                                      "item_id_enc_y":"sim_item_id_enc"})

sim_items.to_parquet("../data/processed/sim_items_covis.parquet", index=False)
print("similar rows", len(sim_items))
sim_items.head()


In [ ]:
# Co-visitation recommendations
# Base interactions for test users
test_users = sorted(events_test["user_id_enc"].unique())
TOPN = 100

base = (
    ui_train.loc[ui_train["user_id_enc"].isin(test_users), ["user_id_enc","item_id_enc"]]
          .drop_duplicates()
)

seed = base.rename(columns={"item_id_enc": "seed_item_id_enc"})

sim = sim_items.rename(columns={
    "item_id_enc": "seed_item_id_enc",
    "sim_item_id_enc": "item_id_enc",
    "score": "co_score"
})

recs = seed.merge(sim, on="seed_item_id_enc", how="left")
recs = recs.dropna(subset=["item_id_enc"])[["user_id_enc","item_id_enc","co_score"]]

# Filter already seen items
seen = base.assign(seen=1)
recs = recs.merge(seen, on=["user_id_enc","item_id_enc"], how="left")
recs = recs[recs["seen"].isna()].drop(columns=["seen"])

# Aggregate and keep top-n items
covis_recs = (
    recs.groupby(["user_id_enc","item_id_enc"])["co_score"].sum().reset_index()
        .sort_values(["user_id_enc","co_score"], ascending=[True, False])
        .groupby("user_id_enc").head(TOPN)
        .rename(columns={"co_score":"score"})
)

print("covis recs shape", covis_recs.shape)

# Metrics
m5  = eval_at_k(covis_recs, test_atc, k=5)
m20 = eval_at_k(covis_recs, test_atc, k=20)
cov_items = (covis_recs.groupby("user_id_enc").head(100)["item_id_enc"].nunique() / n_items)

print("users in eval", m5["users_eval"])
print("precision@5", round(m5["precision@5"], 4),
      "recall@5", round(m5["recall@5"], 4),
      "ndcg@5", round(m5["ndcg@5"], 4),
      "hit_rate", round(m5["hit_rate"], 4))
print("precision@20", round(m20["precision@20"], 4),
      "recall@20", round(m20["recall@20"], 4),
      "ndcg@20", round(m20["ndcg@20"], 4))
print("coverage@100", round(cov_items, 4))

covis_recs.to_parquet("../data/processed/covis_recommendations.parquet", index=False)


In [ ]:
# Global train popularity

w = {"view": 1, "addtocart": 3, "transaction": 5}

pop = events_train.copy()
pop = pop[
    pop["visitorid"].isin(ui_train["visitorid"]) &
    pop["itemid"].isin(items["item_id"])
].copy()

pop["item_id_enc"] = i_enc.transform(pop["itemid"])
pop["w"] = pop["event"].map(w).fillna(0)

top_pop = (pop.groupby("item_id_enc")["w"].sum()
             .sort_values(ascending=False)
             .head(500)
             .index
             .tolist())

u_test = sorted(events_test["user_id_enc"].unique())
top_pop_recs = pd.DataFrame({
    "user_id_enc": np.repeat(u_test, len(top_pop)),
    "item_id_enc": top_pop * len(u_test),
    "score": np.tile(np.arange(len(top_pop), 0, -1), len(u_test))
})

m5  = eval_at_k(top_pop_recs, test_atc, k=5)
m20 = eval_at_k(top_pop_recs, test_atc, k=20)
cov_items = (top_pop_recs.groupby("user_id_enc").head(100)["item_id_enc"].nunique() / n_items)

print("users in eval", m5["users_eval"])
print("precision@5", round(m5["precision@5"], 4),
      "recall@5", round(m5["recall@5"], 4),
      "ndcg@5", round(m5["ndcg@5"], 4))
print("precision@20", round(m20["precision@20"], 4),
      "recall@20", round(m20["recall@20"], 4),
      "ndcg@20", round(m20["ndcg@20"], 4))
print("coverage@100", round(cov_items, 4))

top_pop_recs.to_parquet("../data/processed/top_pop_recommendations.parquet", index=False)


In [ ]:
# Split the test period into labels and inference windows
LABEL_SPLIT = pd.Timestamp("2015-10-01")

# Test split
events_test_labels = events_test[events_test["ts"] < LABEL_SPLIT].copy()
events_test_infer  = events_test[events_test["ts"] >= LABEL_SPLIT].copy()

# Positive events addtocart
atc_labels = (events_test_labels.query("event == 'addtocart'")
              [["user_id_enc","item_id_enc"]].drop_duplicates())
atc_infer  = (events_test_infer.query("event == 'addtocart'")
              [["user_id_enc","item_id_enc"]].drop_duplicates())

print("labels users", atc_labels["user_id_enc"].nunique(),
      "labels pairs", len(atc_labels),
      "| infer users", atc_infer["user_id_enc"].nunique(),
      "infer pairs", len(atc_infer))

# See the project README for context.
als_df   = als_recs.rename(columns={"score":"als_score"})
covis_df = pd.read_parquet("../data/processed/covis_recommendations.parquet") \
              .rename(columns={"score":"co_score"})
pop_df   = pd.read_parquet("../data/processed/top_pop_recommendations.parquet") \
              .rename(columns={"score":"pop_score"})

# See the project README for context.
candidates = als_df.merge(covis_df, on=["user_id_enc","item_id_enc"], how="outer")
candidates = candidates.merge(pop_df, on=["user_id_enc","item_id_enc"], how="outer")
for c in ["als_score","co_score","pop_score"]:
    candidates[c] = candidates[c].fillna(0.0)

print("candidates total", candidates.shape)

# See the project README for context.
labels = atc_labels.copy().assign(target=1)

cand_train = candidates[candidates["user_id_enc"].isin(labels["user_id_enc"].unique())].copy()
cand_train = cand_train.merge(labels, on=["user_id_enc","item_id_enc"], how="left")
cand_train["target"] = cand_train["target"].fillna(0).astype(int)

# See the project README for context.
cand_train = cand_train.groupby("user_id_enc").filter(lambda x: x["target"].sum() > 0)

# See the project README for context.
negatives_per_user = 4
neg_sample = (cand_train.query("target == 0")
              .groupby("user_id_enc", group_keys=False)
              .apply(lambda x: x.sample(min(len(x), negatives_per_user), random_state=0)))
cand_for_train = pd.concat([cand_train.query("target == 1"), neg_sample], ignore_index=True)

print("train rows", len(cand_for_train),
      "users", cand_for_train["user_id_enc"].nunique(),
      "pos", int(cand_for_train["target"].sum()))
cand_for_train.head()


In [ ]:
features = ["als_score","co_score","pop_score"]
target = "target"

train_pool = Pool(cand_for_train[features], label=cand_for_train[target])

cb = CatBoostClassifier(
    iterations=800,
    depth=6,
    learning_rate=0.1,
    loss_function="Logloss",
    verbose=100,
    random_seed=0
)
cb.fit(train_pool)

print("trained")


In [ ]:
# Choose the split boundary by event-time quantile
q = 0.8
label_split = events_test["ts"].quantile(q)

events_test_labels2 = events_test[events_test["ts"] < label_split].copy()
events_test_infer2  = events_test[events_test["ts"] >= label_split].copy()

atc_labels2 = (events_test_labels2.query("event == 'addtocart'")
               [["user_id_enc","item_id_enc"]].drop_duplicates())
atc_infer2  = (events_test_infer2.query("event == 'addtocart'")
               [["user_id_enc","item_id_enc"]].drop_duplicates())

# Fallback if the late window is empty
if atc_infer2.empty:
    q = 0.6
    label_split = events_test["ts"].quantile(q)
    events_test_labels2 = events_test[events_test["ts"] < label_split].copy()
    events_test_infer2  = events_test[events_test["ts"] >= label_split].copy()
    atc_labels2 = (events_test_labels2.query("event == 'addtocart'")
                   [["user_id_enc","item_id_enc"]].drop_duplicates())
    atc_infer2  = (events_test_infer2.query("event == 'addtocart'")
                   [["user_id_enc","item_id_enc"]].drop_duplicates())

print("labels users", atc_labels2["user_id_enc"].nunique(),
      "labels pairs", len(atc_labels2),
      "| infer users", events_test_infer2["user_id_enc"].nunique(),
      "infer pairs", len(atc_infer2))

# Run inference without retraining
users_infer2 = sorted(events_test_infer2["user_id_enc"].unique())
cand_rank2 = (candidates[candidates["user_id_enc"].isin(users_infer2)]
              .drop_duplicates(["user_id_enc","item_id_enc"])
              .copy())

pred = cb.predict_proba(cand_rank2[["als_score","co_score","pop_score"]])[:, 1]
cand_rank2["cb_score"] = pred

final_recs2 = (cand_rank2.sort_values(["user_id_enc","cb_score"], ascending=[True, False])
                          .groupby("user_id_enc").head(100)
                          .rename(columns={"cb_score":"score"}))

print("final recs", final_recs2.shape)

m5  = eval_at_k(final_recs2, atc_infer2, k=5)
m20 = eval_at_k(final_recs2, atc_infer2, k=20)

cov_items2 = (final_recs2.groupby("user_id_enc").head(100)["item_id_enc"].nunique() / n_items)

print("users in eval", m5["users_eval"])
print("precision@5", round(m5["precision@5"], 4),
      "recall@5", round(m5["recall@5"], 4),
      "ndcg@5", round(m5["ndcg@5"], 4),
      "hit_rate", round(m5["hit_rate"], 4))
print("precision@20", round(m20["precision@20"], 4),
      "recall@20", round(m20["recall@20"], 4),
      "ndcg@20", round(m20["ndcg@20"], 4))
print("coverage@100", round(cov_items2, 4))

final_recs2.to_parquet("../data/processed/final_recommendations_cb.parquet", index=False)


In [ ]:
os.makedirs("../models", exist_ok=True)

user_map = pd.DataFrame({
    "user_id_enc": np.arange(len(u_enc.classes_), dtype=int),
    "user_id":     u_enc.classes_.astype(np.int64)
})
item_map = pd.DataFrame({
    "item_id_enc": np.arange(len(i_enc.classes_), dtype=int),
    "item_id":     i_enc.classes_.astype(np.int64)
})

# See the project README for context.
fr = final_recs2.merge(user_map, on="user_id_enc").merge(item_map, on="item_id_enc")
fr = fr[["user_id", "item_id", "score"]].sort_values(["user_id","score"], ascending=[True, False])
fr.to_parquet("../models/final_recommendations_feat.parquet", index=False)
print("saved personal recs", fr.shape)

# See the project README for context.
top_items_enc = (pop.groupby("item_id_enc")["w"].sum().sort_values(ascending=False).head(100)).index.to_numpy()

top_items_raw = i_enc.inverse_transform(top_items_enc)
top_df = pd.DataFrame({"item_id": top_items_raw})
top_df.to_parquet("../models/top_recs.parquet", index=False)
print("saved top_recs", top_df.shape)

# See the project README for context.
max_sim = 50
all_item_ids_enc = np.arange(item_map.shape[0], dtype=int)

sim_ids, sim_scores = als.similar_items(all_item_ids_enc, N=max_sim+1)
sim_df = pd.DataFrame({
    "item_id_enc": np.repeat(all_item_ids_enc, max_sim+1),
    "sim_item_id_enc": sim_ids.flatten(),
    "score": sim_scores.flatten().astype(float)
})
sim_df = sim_df[sim_df["item_id_enc"] != sim_df["sim_item_id_enc"]].copy()

sim_df = sim_df.merge(item_map, on="item_id_enc") \
               .rename(columns={"item_id": "item_id_1"}) \
               .merge(item_map, left_on="sim_item_id_enc", right_on="item_id_enc") \
               .rename(columns={"item_id": "item_id_2"})[["item_id_1","item_id_2","score"]]

sim_df.to_parquet("../models/similar_items.parquet", index=False)
print("saved similar_items", sim_df.shape)


In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5001")  #    5002,   5002
mlflow.set_experiment("ecom_recs")


with mlflow.start_run(run_name="eda"):
    mlflow.log_metrics({
        "rows_events": len(events),
        "users": events["visitorid"].nunique(),
        "items": events["itemid"].nunique(),
        "views": int((events["event"]=="view").sum()),
        "addtocart": int((events["event"]=="addtocart").sum()),
        "transactions": int((events["event"]=="transaction").sum())
    })
    # See the project README for context.
    fig, ax = plt.subplots(1, 2, figsize=(12,4))
    events["event"].value_counts().plot(kind="bar", ax=ax[0], title="Event distribution")
    ax[1].plot(by_day["date"], by_day["events"]); ax[1].set_title("Events by day")
    fig.autofmt_xdate(); plt.tight_layout()
    tmp = Path(tempfile.mkdtemp())
    p = tmp / "eda_fig.png"; fig.savefig(p, bbox_inches="tight"); plt.close(fig)
    mlflow.log_artifact(str(p), artifact_path="figures")
    # See the project README for context.
    nb_path = Path(__file__).resolve() if "__file__" in globals() else Path.cwd() / "ecom_recs.ipynb"
    if nb_path.exists():
        mlflow.log_artifact(str(nb_path), artifact_path="notebooks")

print("MLflow", mlflow.get_tracking_uri())


In [ ]:
with mlflow.start_run(run_name="candidates"):
    mlflow.log_params({"als_factors": 64, "als_iter": 25, "als_reg": 0.05})
    mlflow.log_metrics({
        "als_p_at_5":   m5["precision@5"],
        "als_r_at_5":   m5["recall@5"],
        "als_ndcg_at_5":m5["ndcg@5"],
        "als_p_at_20":   m20["precision@20"],
        "als_r_at_20":   m20["recall@20"],
        "als_ndcg_at_20":m20["ndcg@20"],
    })
    mlflow.log_artifact("../data/processed/als_recommendations.parquet", artifact_path="artifacts")
    mlflow.log_artifact("../data/processed/covis_recommendations.parquet", artifact_path="artifacts")
    mlflow.log_artifact("../data/processed/top_pop_recommendations.parquet", artifact_path="artifacts")
print("MLflow candidates ok")


In [ ]:
cb.save_model("../models/ranker.cbm")

with mlflow.start_run(run_name="ranker_cb_v1"):
    mlflow.log_params({"features": "als_score,co_score,pop_score", "iterations": 800, "depth": 6, "lr": 0.1})
    mlflow.log_metrics({
        "cb_p_at_5":   m5["precision@5"],
        "cb_r_at_5":   m5["recall@5"],
        "cb_ndcg_at_5":m5["ndcg@5"],
        "cb_p_at_20":   m20["precision@20"],
        "cb_r_at_20":   m20["recall@20"],
        "cb_ndcg_at_20":m20["ndcg@20"],
        "coverage_at_100": cov_items2
    })
    mlflow.log_artifact("../models/final_recommendations_feat.parquet", artifact_path="recs")
    mlflow.log_artifact("../models/similar_items.parquet", artifact_path="features")
    mlflow.log_artifact("../models/top_recs.parquet", artifact_path="recs")
    mlflow.log_artifact("../models/ranker.cbm", artifact_path="model")
    try:
        mlflow.catboost.log_model(cb, artifact_path="cb_model", registered_model_name="ecom_ranker_cb")
    except Exception:
        pass
print("MLflow ranker ok")
